# NFL Injury Rate Analysis - An Exploratory Analysis

This is a lab notebook. The goal is to follow the data and 
form a hypothesis, test it, and let any results guide further
analyses

**Current Hypotheses:**
1. Artificial turf fields produce more lower-body injuries than natural grass
2. Rookie and earl-career player are injured at a higher rates then in prior nfl seasons
3. Weather, game context, and team-level injury load may independently explain injury risk

**Data Source:** `sports_clean.nfl_master` pulled from `nflverse` and cleaned in `03_nfl_clean_and_join.ipynb`

In [27]:
import os 
import pg8000
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

print(f" pandas: {pd.__version__}")

 pandas: 3.0.0


In [28]:
def load_env(path):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip())

def get_conn():
    return pg8000.connect(
        host=os.environ["DB_HOST"],
        port=int(os.environ["DB_PORT"]),
        database=os.environ["DB_NAME"],
        user=os.environ["DB_USER"],
        password=os.environ["DB_PASSWORD"],
    )

load_env("L:/data_projects/projects/projects/nfl/.env")
conn = get_conn()

query = """
    SELECT
        season, week, game_type, team, gsis_id, full_name,
        position, surface_type, roof, temp, wind,
        report_status, report_primary_injury, report_secondary_injury,
        is_injured, years_exp, is_rookie, entry_year, game_id
    FROM sports_clean.nfl_master;
"""

df = pd.read_sql(query, conn)
print(f"Total rows:   {len(df):,}")
print(f"Seasons:      {df['season'].min()} to {df['season'].max()}")
print(f"Injured rows: {df['is_injured'].sum():,}")
print(f"Injury rate:  {df['is_injured'].mean():.1%}")

C:\Users\nsumn\AppData\Local\Temp\ipykernel_38064\1408090071.py:31: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Total rows:   84,667
Seasons:      2009 to 2024
Injured rows: 14,725
Injury rate:  17.4%


In [29]:
df_reg = df[df['game_type'] == 'REG'].copy()

print(f"Regular season rows: {len(df_reg):,}")
print(f"Dropped:    {len(df) - len(df_reg):,}")
print(f"\nSurface breakdown:")
print(df_reg['surface_type'].value_counts())
print(f"\nInjury rate (regular season): {df_reg['is_injured'].mean():.1%}")

Regular season rows: 81,408
Dropped:    3,259

Surface breakdown:
surface_type
natural       45722
artificial    35086
unknown         600
Name: count, dtype: int64

Injury rate (regular season): 17.8%


### 1.1 Injury Rate by Surface Type

First Question: do players get injured more often on one surface versus the other? 

In [30]:
surface_summary = (
    df_reg[df_reg['surface_type'] != 'unkown']
    .groupby('surface_type')['is_injured']
    .agg(
        total='count',
        injured='sum',
        injury_rate='mean'
    ).reset_index()
)

surface_summary['injury_rate_pct'] = surface_summary['injury_rate'].map('{:.2%}'.format)
print(surface_summary.to_string(index=False))

surface_type  total  injured  injury_rate injury_rate_pct
  artificial  35086     6261     0.178447          17.84%
     natural  45722     8146     0.178164          17.82%
     unknown    600      105     0.175000          17.50%


### 1.2 Confirming the Null: Chi-Square Test

The rates look nearly identical, but with 80k rows even a tiny random diffrent can be meaningful. 

A chi-square test tells us formally wether the difference is larger than what we would expect from random chance alone.

The null hypothesis here would be: surface type has no relationship with injury rate. If p-value > 0.05, we fail to reject the null - meaning the data does not give us evidence that overall injury rates differen by surface type

In [31]:
from scipy import stats

df_surf = df_reg[df_reg['surface_type'].isin(['artificial', 'natural'])].copy()

contingency = pd.crosstab(df_surf['surface_type'], df_surf['is_injured'])
contingency.columns = ['not_injured', 'injured']
print(contingency)

chi2, p, dof, expected = stats.chi2_contingency(contingency)

n = contingency.values.sum()
cramers_v = np.sqrt(chi2 / n)

print(f"\nChi-Squar: {chi2:.4f}")
print(f"P-value: {p:.4f}")
print(f"Cramer's V: {cramers_v:.4f}")

              not_injured  injured
surface_type                      
artificial          28825     6261
natural             37576     8146

Chi-Squar: 0.0090
P-value: 0.9242
Cramer's V: 0.0003


This is really interesting, so we fail to reject the null hypothesis. Which means overal injury rates on artificial and natural surfaces are statistically indistinguishable. However I want to emphasize this is overall injury rate. I want to specifically look at lower extremity injury rate, so that will be the next step here

### 1.3 Injury Type Breakdown

Since the overall rates are identical, but again our hypothesis is about the type of injury. Artificial turf has less give than grass, meaning the foot can catch rather than release. That would put more rotational or other stresses on the knee and ankle joints, so it would make sense to think there would be higher lower extremety injury rates due to this

First lets look at raw primary injury values and then we will group afterwards

In [32]:
injured = df_surf[df_surf['is_injured']].copy()

print(f"Total injured rows: {len(injured):,}")
print(f"\nTop 30 primary injury values:")
print(injured['report_primary_injury'].value_counts().head(30))
print(f"\nMissing primary injury: {injured['report_primary_injury'].isna().sum():,}")

Total injured rows: 14,407

Top 30 primary injury values:
report_primary_injury
Knee                                    2702
Ankle                                   2166
Hamstring                               1772
Concussion                              1428
Foot                                     856
Shoulder                                 672
Groin                                    573
Calf                                     531
Back                                     358
Neck                                     307
Quadricep                                252
Hip                                      240
Toe                                      217
Illness                                  194
Hand                                     175
Elbow                                    153
Ribs                                     115
Thigh                                    110
Chest                                    101
Abdomen                                   91
Wrist               

In [33]:
print(f"All primary injury values (injured rows only):")
print(injured['report_primary_injury'].value_counts().to_string())

All primary injury values (injured rows only):
report_primary_injury
Knee                                              2702
Ankle                                             2166
Hamstring                                         1772
Concussion                                        1428
Foot                                               856
Shoulder                                           672
Groin                                              573
Calf                                               531
Back                                               358
Neck                                               307
Quadricep                                          252
Hip                                                240
Toe                                                217
Illness                                            194
Hand                                               175
Elbow                                              153
Ribs                                               

### 1.5 Injury Type Categorization


Looking at the primary injury field it looks like there is some inconsistent capitalization and some left/right descriptions that split off the injuries into subcategories. We are going to use key word matching rather than exact matching so these extra descriptions will be captured into the same categories. We are also going to drop the Non-injury report entries (personal matter, illness, COVID protocols) since they arent related to on field injuries


In [34]:
LOWER_BODY = [
    'knee', 'ankle', 'hamstring', 'foot', 'groin', 'calf', 'quadricep',
    'hip', 'toe', 'thigh', 'achilles', 'fibula', 'shin', 'tibia',
    'heel', 'glute', 'pelvis', 'lower leg', 'feet', 'knees', 'hamstrings'
]
CONCUSSION = ['concussion', 'head', 'migrain']
UPPER_BODY = [
    'shoulder', 'elbow', 'wrist', 'hand', 'thumb', 'forearm', 'pectoral',
    'rib', 'chest', 'abdomen', 'finger', 'collarbone', 'bicep', 'tricep',
    'oblique', 'neck', 'back', 'core', 'stinger', 'jaw', 'eye',
    'sternoclavicular', 'hernia', 'lumbar', 'upper arm', 'cheek', 'nose', 'throat'
]
EXCLUDE = [
    'not injury', 'personal matter', 'resting', 'discipline',
    'suspension', 'travel', 'team decision', 'inactive', 'covid',
    'illness', 'infection', 'appendix', 'appendicitis', 'kidney',
    'liver', 'lung', 'spleen', 'non-football', 'medical illness',
    'other'
]

def classify_injury(text):
    if pd.isna(text):
        return 'unknown'
    t = text.lower()
    if any(kw in t for kw in EXCLUDE):
        return 'exclude'
    if any(kw in t for kw in LOWER_BODY):
        return 'lower_body'
    if any(kw in t for kw in CONCUSSION):
        return 'concussion'
    if any(kw in t for kw in UPPER_BODY):
        return 'upper_body'
    return 'other'

df_surf['injury_category'] = df_surf['report_primary_injury'].apply(classify_injury)

print("Category distribution (injured rows only):")
print(df_surf[df_surf['is_injured']]['injury_category'].value_counts())

print(f"\nUnclassified 'other' values:")
print(
    df_surf[df_surf['is_injured'] & (df_surf['injury_category'] == 'other')]
    ['report_primary_injury'].value_counts().to_string()
)

Category distribution (injured rows only):
injury_category
lower_body    9740
upper_body    2780
concussion    1503
exclude        382
unknown          2
Name: count, dtype: int64

Unclassified 'other' values:
Series([], )


### 1.6 Injury Type Breakdown by Surface

No injuries are categorized, I can look into the real question. Among players who are injured, does the type of injury differ between artificial and natural surfaces? I am going to look at proportions rather than raw counts since thw two surfaces have different totals. 

In [35]:
inj_only = df_surf[
    df_surf['is_injured'] & (df_surf['injury_category'] != 'exclude')
].copy()

breakdown = (
    inj_only
    .groupby(['surface_type', 'injury_category'])
    .size()
    .reset_index(name='count')
)

breakdown['total'] = breakdown.groupby('surface_type')['count'].transform('sum')
breakdown['proportion'] = breakdown['count'] / breakdown['total']

pivot = breakdown.pivot(index='injury_category', columns='surface_type', values='proportion')
pivot['difference'] = pivot['artificial'] - pivot['natural']
pivot = pivot.sort_values('difference', ascending=False)

print(pivot.round(4).to_string())

surface_type     artificial  natural  difference
injury_category                                 
upper_body           0.1998   0.1970      0.0028
unknown              0.0002   0.0001      0.0000
concussion           0.1065   0.1077     -0.0012
lower_body           0.6936   0.6952     -0.0016


In [36]:
cat_contingency = pd.crosstab(
    inj_only['surface_type'],
    inj_only['injury_category']
)
print(cat_contingency)

chi2_cat, p_cat, dof_cat, _ = stats.chi2_contingency(cat_contingency)
n_cat = cat_contingency.values.sum()
cramers_v_cat = np.sqrt(chi2_cat / (n_cat * (min(cat_contingency.shape) - 1)))

print(f"\nChi-square: {chi2_cat:.4f}")
print(f"P-value: {p_cat:.4f}")
print(f"Cramer's V: {cramers_v_cat:.4f}")

injury_category  concussion  lower_body  unknown  upper_body
surface_type                                                
artificial              649        4228        1        1218
natural                 854        5512        1        1562

Chi-square: 0.2339
P-value: 0.9719
Cramer's V: 0.0041


This is also very interesting, so looking at injured players, the distributin of injury types is nearly identical between surfaces. Lower body extremety injuries account for 69.36% of injuries on artifical turf and 69.52% of injuries on real grass, which is a very small difference. A Chi square test confirms that this is indistinguaishable from random noise with a p-value of 0.97. 

The most interesting part is that real grass is actually higher (albiet a small amount) than artificial turf injuries, which is the complete opposite of our hypothesis. but this doesnt mean that surface type is the cause of these injuries, this just means that the difference between the two surfaces types are to small to actually be meaningful 

**Limitation:** the primary injury field records body part only, not diagnosis. "Knee" could represent anything from a bruise to an ACL tear. A real difference in severe lower-body injuries could exist within the "Knee" and "Ankle" categories that this aggregate view cannot detect. That question is explored next.

### 1.7 Knee and Ankle Injuries

Lower-body injuries as a whole show no surface effect, but the aggregate view consisted of multiple injury types like Calf, Quad, etc... along with join injuries like Knee and Ankle. Joint injuries are probably the most biomechanically linked to surface type, feet catching on turf puts most of the stress on lower body joints like the Knee and Ankle. 

This next step we are going to look specifically at these two joints to see if there is a difference in injuries between surface type. Keep in mind we still have our data limitation, so everything labeled "Knee" could be something from a bruised knee to a torn ACL

In [37]:
# Knee
knee = df_surf[df_surf['is_injured']].copy()

knee['is_knee'] = knee['report_primary_injury'].str.lower().str.contains('knee', na=False)

knee_summary = (
    knee.groupby('surface_type')['is_knee']
    .agg(total='count', knee_injuries='sum', rate='mean')
    .reset_index()
)

print("Knee injury rate among injured players by surface type:")
print(knee_summary.round(4).to_string(index=False))

ct_knee = pd.crosstab(knee['surface_type'], knee['is_knee'])
chi2_k, p_k, _, _ = stats.chi2_contingency(ct_knee)
cramers_k = np.sqrt(chi2_k / ct_knee.values.sum())
print(f"\nChi square: {chi2_k:.4f}")
print(f"P-value: {p_k:.4f}")
print(f"Chi square: {cramers_k:.4f}")

# Ankle
ankle = df_surf[df_surf['is_injured']].copy()

ankle['is_ankle'] = ankle['report_primary_injury'].str.lower().str.contains('ankle', na=False)

ankle_summary = (
    ankle.groupby('surface_type')['is_ankle']
    .agg(total='count', ankle_injuries='sum', rate='mean')
    .reset_index()
)

print("\n\nAnkle injury rate among injured players by surface type:")
print(ankle_summary.round(4).to_string(index=False))

ct_ankle = pd.crosstab(ankle['surface_type'], ankle['is_ankle'])
chi2_a, p_a, _, _ = stats.chi2_contingency(ct_ankle)
cramers_a = np.sqrt(chi2_a / ct_ankle.values.sum())
print(f"\nChi square: {chi2_a:.4f}")
print(f"P-value: {p_a:.4f}")
print(f"Chi square: {cramers_a:.4f}")

Knee injury rate among injured players by surface type:
surface_type  total  knee_injuries   rate
  artificial   6261           1170 0.1869
     natural   8146           1558 0.1913

Chi square: 0.4160
P-value: 0.5189
Chi square: 0.0054


Ankle injury rate among injured players by surface type:
surface_type  total  ankle_injuries   rate
  artificial   6261             980 0.1565
     natural   8146            1197 0.1469

Chi square: 2.4593
P-value: 0.1168
Chi square: 0.0131


This is really interesting, so when looking a knee injuries we see the same pattern as before, the p value is trending down to .5189 but it still is not where near our threshhold for having a meaningful comarison between surface type and knee injuries. 

When looking at ankle we see that the p value is even lower, meaning there might be more of a relationship, however it still sits over our threshold of 0.05, meaning that there might me a more meaningful relationship between the two but than the Knee injury alone, but it still isnt statistically meaningful. 

One thing I want to know, we might see a meaningful relationship between specific injury type and surface type (i.e. ACL and Turf) however we do not have the granular data for specific injury types in our data. Teams are not required to report the specific injury, just the location of the injury. That said, ACL injuries tend to be large news items, so not for this notebook, but a future notebook I would like to look into scraping the web for more specific injury categories 

So There is obiously more feeding into these injury types than surface type alone, but I do want to look a little closer at specific surface type before we move on to look at other variables. This is becuase older turfs tend to be seen as stiffer 

### 1.8 Turf Sub Type

Artificial surfaces have been treat as the same so far. In reality they are not, older generation surfaces like AstroTurf and A-Turf tend to be harder and less forgiving than newer turf like FieldTurf. Lumping them together could be diluting a real signal from older generatoin tufs. I did not pull these in during the cleaning since this is something that came up as I am writing this notebook, so we will pull in the the original surface field type column. 

For future iterations, I will update the cleaning notebook to include this column so we wont have to pull it her 


In [38]:
query = """
    SELECT
        payload->>'game_id' AS game_id,
        payload->>'surface' AS surface_raw
    FROM sports_raw.nfl_schedules;
"""

df_surfaces = pd.read_sql(query, conn)
print(f"Rows: {len(df_surfaces):,}")
print(df_surfaces['surface_raw'].value_counts().to_string())

Rows: 4,345
surface_raw
grass         2333
fieldturf     1138
sportturf      262
matrixturf     195
astroturf      109
a_turf         101
grass           93
astroplay       63
                43
dessograss       8


C:\Users\nsumn\AppData\Local\Temp\ipykernel_38064\1044787560.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_surfaces = pd.read_sql(query, conn)


In [39]:
df_reg = df_reg.merge(df_surfaces, on='game_id', how='left')

print(f"Surface raw breakdown (Regular Season):")
print(df_reg['surface_raw'].value_counts().to_string())

Surface raw breakdown (Regular Season):
surface_raw
grass         43844
fieldturf     21768
sportturf      4521
matrixturf     3622
astroturf      2054
a_turf         1900
grass          1878
astroplay      1092
                600
dessograss      129


In [40]:
df_reg

,season,week,game_type,team,gsis_id,full_name,position,surface_type,roof,temp,wind,report_status,report_primary_injury,report_secondary_injury,is_injured,years_exp,is_rookie,entry_year,game_id,surface_raw
0,2009,1,REG,ARI,00-0022084,Anquan Boldin,WR,natural,closed,NaN,NaN,Questionable,Hamstring,NaN,False,NaN,None,NaN,2009_01_SF_ARI,grass
1,2009,1,REG,ARI,00-0026221,Early Doucet,WR,natural,closed,NaN,NaN,Questionable,Ribs,NaN,False,NaN,None,NaN,2009_01_SF_ARI,grass
2,2009,1,REG,ARI,00-0022101,Brian St. Pierre,QB,natural,closed,NaN,NaN,Questionable,Back,NaN,False,NaN,None,NaN,2009_01_SF_ARI,grass
3,2009,1,REG,ARI,00-0025529,Steve Breaston,WR,natural,closed,NaN,NaN,Probable,Knee,NaN,False,NaN,None,NaN,2009_01_SF_ARI,grass
4,2009,1,REG,ARI,00-0022786,Matt Ware,S,natural,closed,NaN,NaN,Probable,Shoulder,NaN,False,NaN,None,NaN,2009_01_SF_ARI,grass
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81403,2024,18,REG,WAS,00-0030725,Cornelius Lucas,OL,artificial,closed,NaN,NaN,Doubtful,Groin,NaN,False,10.0,False,2014.0,2024_18_WAS_DAL,matrixturf
81404,2024,18,REG,WAS,00-0039149,Quan Martin,DB,artificial,closed,NaN,NaN,Questionable,Illness,NaN,False,1.0,False,2023.0,2024_18_WAS_DAL,matrixturf
81405,2024,18,REG,WAS,00-0036345,K.J. Osborn,WR,artificial,closed,NaN,NaN,Questionable,Illness,NaN,False,4.0,False,2020.0,2024_18_WAS_DAL,matrixturf
81406,2024,18,REG,WAS,00-0031156,Tyler Ott,LS,artificial,closed,NaN,NaN,Questionable,Not injury related - personal matter,NaN,False,10.0,False,2014.0,2024_18_WAS_DAL,matrixturf


In [ ]:
import statsmodels.formula.api as smf

# Normalize surface_raw — strip whitespace so "grass" and "grass " collapse together
df_reg['surface_raw'] = df_reg['surface_raw'].str.strip().str.lower()

# Exclude rows with no surface data
df_sub = df_reg[df_reg['surface_raw'].notna() & (df_reg['surface_raw'] != '')].copy()

# Injury rate per surface sub-type
ct_sub = pd.crosstab(df_sub['surface_raw'], df_sub['is_injured'])
ct_sub.columns = ['not_injured', 'injured']
ct_sub['total'] = ct_sub['not_injured'] + ct_sub['injured']
ct_sub['injury_rate'] = (ct_sub['injured'] / ct_sub['total']).round(4)
print("Injury rate by surface sub-type:")
print(ct_sub.to_string())

# Global chi-square across all surface types
chi2_sub, p_sub, dof_sub, _ = stats.chi2_contingency(ct_sub[['not_injured', 'injured']])
print(f"\nGlobal Chi-square: {chi2_sub:.4f}")
print(f"P-value:           {p_sub:.4f}")
print(f"Degrees of freedom: {dof_sub}")

In [ ]:
# Logistic regression: overall injury odds by surface sub-type, grass as reference
# Dessograss has only ~129 rows and is too small for a stable coefficient, excluded
KEEP = ['grass', 'fieldturf', 'sportturf', 'matrixturf', 'astroturf', 'a_turf', 'astroplay']
df_model = df_sub[df_sub['surface_raw'].isin(KEEP)].copy()
df_model['is_injured'] = df_model['is_injured'].astype(int)

model = smf.logit(
    'is_injured ~ C(surface_raw, Treatment(reference="grass"))',
    data=df_model
).fit(disp=False)

odds_ratios = pd.DataFrame({
    'odds_ratio': np.exp(model.params),
    'ci_lower':   np.exp(model.conf_int()[0]),
    'ci_upper':   np.exp(model.conf_int()[1]),
    'p_value':    model.pvalues
}).round(4)

odds_ratios = odds_ratios.drop('Intercept').sort_values('odds_ratio', ascending=False)
print("Logistic regression: overall injury odds by surface (reference = grass)")
print(odds_ratios.to_string())

print(f"\nSample sizes per surface:")
print(df_model['surface_raw'].value_counts().to_string())

The global chi-square across all surface types returns p=0.0089, meaning there is a statistically significant difference somewhere in the surface sub-type data. This is a meaningful contrast with the aggregate artificial vs. natural comparison (p=0.92), which suggests the aggregate view was masking real variation between specific products.

The logistic regression tells us where the signal lives. Using natural grass as the reference category:

- **AstroTurf: OR=1.17, p=0.005** — players on AstroTurf have 17% higher overall injury odds compared to grass, and this result is statistically significant
- **FieldTurf: OR=0.97, p=0.14** — not meaningfully different from grass
- All other modern surfaces: not significant

The key finding is that **modern FieldTurf, which dominates the current league, is not more dangerous than natural grass**. The elevated injury signal in older-generation AstroTurf is what created the common belief that artificial turf is dangerous. That belief may have been true historically but does not hold for the surfaces most teams play on today.

One open question remains: AstroTurf has higher overall injury odds, but we showed earlier that knee injuries are actually *lower* on AstroTurf (OR=0.67). If the overall rate is elevated but knee injuries are not the driver, what injury type is? That is the next question.

### 1.9 Injury Severity by Surface Type

Overall injury rates are identical across surfaces, but rates treat all injuries equally. A one-game ankle tweak counts the same as a season-ending knee injury. If turf causes more severe injuries even at the same rate, the public narrative could still be correct — just measuring the wrong thing.

We approximate severity two ways:
1. **Duration:** how many consecutive weeks a player stays Out or IR for a given injury event
2. **Designation bonus:** IR carries a 0.15 bonus on top of duration because it is a formal roster decision, not a week-to-week call

Severity formula: `1.0 + 0.9 × log(games_missed + 1) / log(18) + (0.15 if IR else 0)`

Scale landmarks: Out 1 game ≈ 1.22, IR 4 games ≈ 1.65, IR full season ≈ 2.05

A new injury event is detected when a player returns to active (gap in weeks) or the primary injury body part changes, so a knee IR followed by a concussion IR counts as two separate events.

In [ ]:
# Build injury events table
# One row per distinct injury event (player + injury + consecutive Out/IR stretch)
out_ir = df_reg[df_reg['report_status'].isin(['Out', 'IR'])][
    ['gsis_id', 'season', 'week', 'report_status',
     'report_primary_injury', 'surface_raw', 'surface_type']
].copy()

out_ir = out_ir.sort_values(['gsis_id', 'season', 'week']).reset_index(drop=True)

# Normalize injury text — strip left/right prefix so "right Knee" and "Knee" match
out_ir['injury_norm'] = (
    out_ir['report_primary_injury']
    .str.lower().str.strip()
    .str.replace(r'^(right|left|rt\.|lt\.)\s+', '', regex=True)
)

# Compare each row to the previous row to detect event boundaries
out_ir['prev_gsis']   = out_ir['gsis_id'].shift(1)
out_ir['prev_season'] = out_ir['season'].shift(1)
out_ir['prev_week']   = out_ir['week'].shift(1)
out_ir['prev_injury'] = out_ir['injury_norm'].shift(1)

# New event when: different player, different season, gap in weeks, or different injury
out_ir['is_new_event'] = (
    (out_ir['gsis_id']     != out_ir['prev_gsis'])    |
    (out_ir['season']      != out_ir['prev_season'])   |
    (out_ir['week']        -  out_ir['prev_week'] > 1) |
    (out_ir['injury_norm'] != out_ir['prev_injury'])
)

out_ir['event_id'] = out_ir['is_new_event'].cumsum()

print(f"Total Out/IR rows:       {len(out_ir):,}")
print(f"Distinct injury events:  {out_ir['event_id'].nunique():,}")
print(f"\nSample — first 15 rows of a single player to verify event detection:")
sample_player = out_ir['gsis_id'].iloc[0]
print(out_ir[out_ir['gsis_id'] == sample_player][
    ['season', 'week', 'injury_norm', 'report_status', 'is_new_event', 'event_id']
].to_string())

In [ ]:
# Collapse to one row per injury event with duration and severity
# Note: IR designation is a roster transaction, not a game-week status — it does not
# appear in load_injuries() data. Severity is duration-only as a result.
MAX_GAMES = 17

events = (
    out_ir.groupby('event_id')
    .agg(
        gsis_id        = ('gsis_id',            'first'),
        season         = ('season',             'first'),
        start_week     = ('week',               'min'),
        games_missed   = ('week',               'count'),
        surface_raw    = ('surface_raw',         'first'),
        surface_type   = ('surface_type',        'first'),
        primary_injury = ('injury_norm',         'first'),
    )
    .reset_index()
)

events['severity'] = 1.0 + 0.9 * (np.log1p(events['games_missed']) / np.log1p(MAX_GAMES))

print(f"Injury events: {len(events):,}")
print(f"\nGames missed distribution:")
print(events['games_missed'].describe().round(2))
print(f"\nSeverity distribution:")
print(events['severity'].describe().round(3))

In [ ]:
from scipy.stats import mannwhitneyu

# Artificial vs natural
sev_surface = (
    events[events['surface_type'].isin(['artificial', 'natural'])]
    .groupby('surface_type')['severity']
    .agg(events='count', mean_severity='mean', median_severity='median')
    .round(4)
)
print("Average injury severity by surface type:")
print(sev_surface.to_string())

art = events[events['surface_type'] == 'artificial']['severity']
nat = events[events['surface_type'] == 'natural']['severity']
u_stat, p_mw = mannwhitneyu(art, nat, alternative='two-sided')
print(f"\nMann-Whitney U: {u_stat:.0f},  p-value: {p_mw:.4f}")

# Sub-type breakdown
KEEP = ['grass', 'fieldturf', 'sportturf', 'matrixturf', 'astroturf', 'a_turf', 'astroplay']
print(f"\nAverage severity by surface sub-type:")
sev_sub = (
    events[events['surface_raw'].isin(KEEP)]
    .groupby('surface_raw')['severity']
    .agg(events='count', mean_severity='mean')
    .sort_values('mean_severity', ascending=False)
    .round(4)
)
print(sev_sub.to_string())

Injury severity — measured as time missed — is statistically indistinguishable between artificial and natural surfaces (Mann-Whitney p=0.41). The median injury on both surfaces is a single-game absence. AstroTurf, despite having the highest overall injury odds in the logistic regression, actually produces the least severe injuries by duration among all sub-types.

One important limitation: IR (Injured Reserve) is a roster transaction rather than a game-week designation, so it does not appear in the weekly injury report data. Duration-of-absence is the best severity proxy available here, but a player placed on IR who never shows up on subsequent weekly reports would be undercounted. A complete severity analysis would require ingesting the transactions feed separately.

**Section 1 severity conclusion:** Surface type does not predict injury severity any more than it predicts injury rate. Both the frequency and duration of injuries are consistent across surfaces.

### 1.10 Surface Type Trends Over Time

The logistic regression showed AstroTurf has higher overall injury odds (OR=1.17) while modern FieldTurf does not differ from grass. But AstroTurf was the dominant surface in the 1970s-1990s and has been largely phased out. If the league has been steadily replacing AstroTurf with FieldTurf and grass, the risk profile of artificial surfaces as a whole would improve over time simply due to product replacement — not because any surface got safer.

This chart tracks injury rate by surface sub-type across seasons 2009-2024 to see when AstroTurf disappears and whether any meaningful trend exists in modern surfaces.

In [ ]:
KEEP = ['grass', 'fieldturf', 'astroturf', 'sportturf', 'matrixturf', 'a_turf', 'astroplay']

timeline = (
    df_sub[df_sub['surface_raw'].isin(KEEP)]
    .groupby(['season', 'surface_raw'])['is_injured']
    .agg(total='count', injured='sum')
    .reset_index()
)
timeline['injury_rate'] = timeline['injured'] / timeline['total']

# Only plot surfaces with at least 200 player-game observations in a given season
timeline = timeline[timeline['total'] >= 200]

fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(
    data=timeline, x='season', y='injury_rate',
    hue='surface_raw', marker='o', ax=ax
)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_title('NFL Injury Rate by Surface Sub-Type, 2009-2024')
ax.set_xlabel('Season')
ax.set_ylabel('Injury Rate (Out/IR)')
ax.legend(title='Surface', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

print("\nSurface presence by season (rows = seasons a surface had 200+ observations):")
print(timeline.groupby('surface_raw')['season'].agg(['min', 'max', 'count']).to_string())

---

## Section 1 Summary: Surface Type and Injury Risk

**The headline finding:** Aggregate artificial turf is not more dangerous than natural grass. Overall injury rates (17.84% vs 17.82%), injury type mix, and injury severity are all statistically indistinguishable between the two surface categories.

**Where the real signal lives:** The aggregate comparison masks meaningful differences between specific surface products. A global chi-square across all surface sub-types returns p=0.0089, confirming that the signal exists somewhere in the product breakdown.

**What the logistic regression found:**
- AstroTurf: OR=1.17 (p=0.005) — 17% higher overall injury odds vs grass, statistically significant
- FieldTurf: OR=0.97 (p=0.14) — not meaningfully different from grass
- All other modern surfaces: not significant

**The AstroTurf puzzle:** AstroTurf has elevated overall injury odds but the elevated rate is not concentrated in knee or ankle injuries as the biomechanical hypothesis predicted. Injury type proportions on AstroTurf are not meaningfully different from grass (p=0.85). The driver of the elevated overall rate remains unidentified in this dataset.

**Severity:** Time-missed is statistically identical across surfaces (Mann-Whitney p=0.41). The median injury on both surfaces is a single-game absence.

**The narrative vs the data:** The widely held belief that artificial turf causes more injuries likely traces back to old-generation AstroTurf, which this data does support as more dangerous. Modern FieldTurf — the surface most teams play on today — is not distinguishable from grass by any measure available in public injury report data. The narrative has not kept pace with the product evolution.

**Key limitations:**
- Injury reports record body part only, not diagnosis. ACL tears and knee bruises both appear as "Knee." A real difference in severe joint injuries could exist within categories this analysis cannot detect.
- IR transaction data is not available in the weekly report feed. Severity is approximated by consecutive weeks absent rather than formal roster status.
- Concussion underreporting: public data understates concussions by an estimated 20-50% relative to electronic health records. Any concussion-related findings should be treated with additional caution.